# 06 - Fuzzy Matching

## Objective

Menguji fuzzy similarity hanya pada candidate pair hasil deterministic blocking. Tidak dilakukan perbandingan seluruh dataset.

## Research Questions

1. Bagaimana distribusi skor similarity pada candidate pair deterministic?
2. Bagaimana jumlah kandidat berubah pada threshold 80, 90, dan 95?
3. Apakah skor multi-field lebih informatif daripada satu field saja?
4. Apakah threshold tertentu dapat dipilih tanpa ground truth?

## Hypothesis

- Candidate pair yang didukung beberapa exact rule cenderung memiliki similarity nama dan alamat lebih tinggi.
- Threshold tinggi akan mengurangi kandidat, tetapi dapat menurunkan recall.
- Tanpa ground truth, threshold hanya dapat dibandingkan sebagai sensitivity analysis, bukan dipilih sebagai threshold final.

## Scope and limitations

- Fuzzy scoring hanya diterapkan pada candidate pair hasil blocking R1-R4.
- Tidak ada N x N comparison.
- Tidak ada automatic merge atau penghapusan baris.
- `customer_id` hanya dipakai untuk audit internal, bukan sebagai label ground truth.
- Nilai customer tidak ditampilkan; output disimpan sebagai row index dan agregat skor.

In [ ]:
from itertools import combinations
from pathlib import Path
from time import perf_counter
import pandas as pd
from rapidfuzz import fuzz

DATA_CANDIDATES = [
    Path.cwd() / 'data' / 'processed' / 'crm_50000_customers_standardized.csv',
    Path.cwd().parent / 'data' / 'processed' / 'crm_50000_customers_standardized.csv',
]
DATA_PATH = next((path.resolve() for path in DATA_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Dataset terstandardisasi tidak ditemukan.')

df = pd.read_csv(DATA_PATH).reset_index(names='row_index')
print('File:', DATA_PATH)
print('Shape:', df.shape)

## Experiment 1 - Rebuild deterministic candidate pairs

Candidate pair direkonstruksi dari aturan R1-R4 agar notebook ini reproducible dan tidak bergantung pada state notebook 05.

In [ ]:
name_dob_parts = df[['name_key_std', 'dob_std']]
df['name_dob_key'] = name_dob_parts.fillna('').astype('string').agg('|'.join, axis=1)
df.loc[name_dob_parts.isna().any(axis=1), 'name_dob_key'] = pd.NA
email_phone_parts = df[['email_std', 'phone_digits_std']]
df['email_phone_key'] = email_phone_parts.fillna('').astype('string').agg('|'.join, axis=1)
df.loc[email_phone_parts.isna().any(axis=1), 'email_phone_key'] = pd.NA

pair_rules = {}
rule_definitions = [
    ('email_std', 'R1_exact_email'),
    ('phone_digits_std', 'R2_exact_phone'),
    ('name_dob_key', 'R3_name_plus_dob'),
    ('email_phone_key', 'R4_email_plus_phone'),
]
for key, rule in rule_definitions:
    for _, block in df.dropna(subset=[key]).groupby(key, sort=False):
        row_indices = sorted(block['row_index'].tolist())
        for left_index, right_index in combinations(row_indices, 2):
            pair_rules.setdefault((left_index, right_index), set()).add(rule)

candidate_pairs = pd.DataFrame([
    {
        'left_row_index': left_index,
        'right_row_index': right_index,
        'supporting_rules': '|'.join(sorted(rules)),
        'rule_count': len(rules),
    }
    for (left_index, right_index), rules in sorted(pair_rules.items())
])
print('Candidate pairs:', len(candidate_pairs))

## Experiment 2 - Multi-field fuzzy score

Skor menggunakan `fuzz.ratio` pada nama, alamat, dan kota terstandardisasi. Nilai akhir hanya dihitung jika minimal dua field tersedia pada kedua row agar skor antar-pair dapat dibandingkan secara lebih adil.

Threshold 80, 90, dan 95 dipakai sebagai sensitivity analysis. Angka tersebut bukan threshold final karena ground truth belum tersedia.

In [ ]:
def text_value(value):
    if pd.isna(value):
        return None
    return str(value)

def pair_score(left_row, right_row):
    scores = {}
    for field in ['name_key_std', 'address_std', 'city_std']:
        left_value = text_value(left_row[field])
        right_value = text_value(right_row[field])
        if left_value is not None and right_value is not None:
            scores[field] = fuzz.ratio(left_value, right_value)
    if not scores:
        return pd.Series({'name_score': pd.NA, 'address_score': pd.NA, 'city_score': pd.NA, 'field_count': 0, 'mean_score': pd.NA})
    field_count = len(scores)
    mean_score = sum(scores.values()) / field_count if field_count >= 2 else pd.NA
    return pd.Series({
        'name_score': scores.get('name_key_std', pd.NA),
        'address_score': scores.get('address_std', pd.NA),
        'city_score': scores.get('city_std', pd.NA),
        'field_count': field_count,
        'mean_score': mean_score,
    })

start_time = perf_counter()
score_rows = []
for pair in candidate_pairs.itertuples(index=False):
    left_row = df.iloc[pair.left_row_index]
    right_row = df.iloc[pair.right_row_index]
    scores = pair_score(left_row, right_row)
    score_rows.append({
        'left_row_index': pair.left_row_index,
        'right_row_index': pair.right_row_index,
        'supporting_rules': pair.supporting_rules,
        'rule_count': pair.rule_count,
        **scores.to_dict(),
    })
fuzzy_scores = pd.DataFrame(score_rows)
runtime_seconds = perf_counter() - start_time
print('Scored pairs:', len(fuzzy_scores))
print(f'Runtime seconds: {runtime_seconds:.3f}')
fuzzy_scores[['name_score', 'address_score', 'city_score', 'field_count', 'mean_score']].describe().round(2)

In [ ]:
thresholds = [80, 90, 95]
threshold_rows = []
for threshold in thresholds:
    selected = fuzzy_scores[
        fuzzy_scores['field_count'].ge(2)
        & fuzzy_scores['mean_score'].ge(threshold)
    ]
    threshold_rows.append({
        'threshold': threshold,
        'candidate_pairs_above_threshold': len(selected),
        'percentage_of_scored_pairs': len(selected) / fuzzy_scores['mean_score'].notna().sum() * 100,
        'pairs_with_2_or_more_rules': int((selected['rule_count'] >= 2).sum()),
    })
threshold_summary = pd.DataFrame(threshold_rows)
threshold_summary.round(2)

In [ ]:
rule_score_summary = (
    fuzzy_scores.groupby('rule_count')
    .agg(pair_count=('mean_score', 'size'), mean_score=('mean_score', 'mean'), median_score=('mean_score', 'median'))
    .reset_index()
)
rule_score_summary.round(2)

## Experiment 3 - Internal audit by threshold

Audit ini membandingkan hasil threshold dengan `customer_id` hanya sebagai consistency check. Ini bukan evaluasi ground truth.

In [ ]:
audit_base = fuzzy_scores.merge(df[['row_index', 'customer_id']], left_on='left_row_index', right_on='row_index', how='left').rename(columns={'customer_id': 'left_customer_id'}).drop(columns='row_index')
audit_base = audit_base.merge(df[['row_index', 'customer_id']], left_on='right_row_index', right_on='row_index', how='left').rename(columns={'customer_id': 'right_customer_id'}).drop(columns='row_index')
audit_base['same_customer_id'] = audit_base['left_customer_id'] == audit_base['right_customer_id']
audit_rows = []
for threshold in thresholds:
    selected = audit_base[audit_base['mean_score'] >= threshold]
    audit_rows.append({
        'threshold': threshold,
        'selected_pairs': len(selected),
        'same_customer_id': int(selected['same_customer_id'].sum()),
        'different_customer_id': int((~selected['same_customer_id']).sum()),
        'same_customer_id_percentage': selected['same_customer_id'].mean() * 100 if len(selected) else 0,
    })
threshold_audit = pd.DataFrame(audit_rows)
threshold_audit.round(2)

In [ ]:
OUTPUT_DIR = DATA_PATH.parents[1] / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SCORE_OUTPUT_PATH = OUTPUT_DIR / 'deterministic_candidate_fuzzy_scores.csv'
fuzzy_scores.to_csv(SCORE_OUTPUT_PATH, index=False)
print('Saved score table:', SCORE_OUTPUT_PATH)
print('Raw dataset still exists:', (DATA_PATH.parents[1] / 'raw' / 'crm_50000_customers_dirty_v3.csv').exists())

# Result, Analysis, and Decision

## Result aktual

Gunakan `threshold_summary`, `rule_score_summary`, dan `threshold_audit` sebagai sumber hasil aktual setelah notebook dijalankan. `field_count` menunjukkan jumlah field yang tersedia pada kedua row; pair dengan kurang dari dua field tidak memperoleh `mean_score` dan tidak masuk threshold summary.

## Analysis

- Candidate pair dibatasi oleh deterministic rules R1-R4, sehingga tidak ada perbandingan N x N.
- Skor hanya comparable untuk pair dengan minimal dua field pembanding.
- Perubahan jumlah pair pada threshold 80, 90, dan 95 adalah sensitivity analysis, bukan pemilihan threshold final.
- Kesamaan `customer_id` tetap hanya consistency check, bukan ground truth dan bukan precision/recall.

## Decision

Score table dapat digunakan untuk audit kandidat, tetapi belum untuk automatic merge. Candidate generation perlu diuji terpisah pada tahap 07 dengan ukuran block, candidate reduction, runtime, dan memory yang terdokumentasi.

Tidak ada merge, delete, atau perubahan raw dataset. Score table disimpan terpisah.

## Next Experiment

Lanjutkan ke `07_blocking_candidate_generation.ipynb` untuk membandingkan blocking strategy dan risiko candidate yang terbuang.